In [85]:
import copy
import gzip
import os
from pathlib import Path
import re
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from Bio import SeqIO, Entrez

cwd = os.getcwd()
if cwd.endswith('Halocins'):
    os.chdir('../..')
    cwd = os.getcwd()

In [39]:
sns.set_palette('colorblind')
sns.set_style('whitegrid')
sns.set_context('paper', font_scale=1.8)
plt.rcParams['font.family'] = 'Helvetica'

palette = sns.color_palette().as_hex()

data_folder = Path('data/')
assert data_folder.is_dir()

amp_db_folder = data_folder / 'amp_db'
assert amp_db_folder.is_dir()

halocins_folder = amp_db_folder / 'Halocins'
assert halocins_folder.is_dir()

halh4_folder = halocins_folder / 'Halocin_H4'
assert halh4_folder.is_dir()

halh4_out = Path('data/outputs/amp_db/Halocins/Halocin_H4')


## Halocin H4 dataset prepatation

In [40]:
hits_df = pd.read_csv(halocins_folder / 'iterative_gtdb_search' / 'iterative_x6_results.tsv', sep='\t')
hits_df = hits_df[hits_df['query'] == 'Halocin_H4'].reset_index(drop=True)

hits_df['domain'] = hits_df['taxlineage'].apply(lambda v: v.split(';')[0].replace('d_', ''))
hits_df['gtdb_phylum'] = hits_df['taxlineage'].apply(lambda v: v.split(';')[1].replace('p_', ''))
hits_df['gtdb_class'] = hits_df['taxlineage'].apply(lambda v: v.split(';')[2].replace('c_', ''))
hits_df['gtdb_order'] = hits_df['taxlineage'].apply(lambda v: v.split(';')[3].replace('o_', ''))
hits_df['gtdb_family'] = hits_df['taxlineage'].apply(lambda v: v.split(';')[4].replace('f_', ''))
hits_df['gtdb_genus'] = hits_df['taxlineage'].apply(lambda v: v.split(';')[5].replace('g_', ''))
hits_df['gtdb_species'] = hits_df['taxlineage'].apply(lambda v: v.split(';')[6].replace('s_', ''))
hits_df = hits_df.drop(columns=['taxlineage'])

hits_df = hits_df.sort_values(['query', 'bits'], ascending=[True, False]).reset_index(drop=True)

hits_df.to_csv(halh4_out / 'Halocin_H4_GTDB_hits.csv', index=False)

hits_df.head()

,query,target,evalue,bits,tstart,tend,domain,gtdb_phylum,gtdb_class,gtdb_order,gtdb_family,gtdb_genus,gtdb_species
0,Halocin_H4,NC_017943.1_248,7.295000e-233,723,1,359,Archaea,Halobacteriota,Halobacteria,Halobacteriales,Haloferacaceae,Haloferax,Haloferax mediterranei
1,Halocin_H4,NZ_AOIP01000056.1_97,2.720000e-100,340,48,347,Archaea,Halobacteriota,Halobacteria,Halobacteriales,Natrialbaceae,Natrialba,Natrialba aegyptia
2,Halocin_H4,NZ_AOIJ01000063.1_217,2.408000e-93,320,1,276,Archaea,Halobacteriota,Halobacteria,Halobacteriales,Natrialbaceae,Natrinema,Natrinema gari
3,Halocin_H4,NZ_AOIN01000078.1_1,1.911000e-42,172,6,156,Archaea,Halobacteriota,Halobacteria,Halobacteriales,Natrialbaceae,Natrialba,Natrialba chahannaoensis
4,Halocin_H4,JAATFT010000037.1_33,2.560000e-32,142,1,210,Bacteria,Bacillota A,Clostridia,Lachnospirales,Lachnospiraceae,Anaerocolumna,Anaerocolumna sp015655075


In [41]:
halh4_protein_ids = set(hits_df['target'].unique())

halh4_records = []
for record in SeqIO.parse(halocins_folder / 'iterative_gtdb_search' / 'iterative_x6_results.fasta', 'fasta'):
    if record.id in halh4_protein_ids:
        halh4_records.append(record)

print(len(halh4_records))

with (halh4_out / 'Halocin_H4_GTDB_hits.fasta').open('w') as f_out:
    SeqIO.write(halh4_records, f_out, 'fasta')

31


### Deduplication

In [42]:
def parse_tblout(path):
    return pd.read_csv(
        path, 
        sep='\s+', 
        comment='#', 
        header=None,
        usecols=[0, 2, 4, 5],
        names=[
            'target', 'query', 'evalue', 'bitscore',
        ],
    )

In [43]:
all_vs_all = parse_tblout(halh4_folder / 'halh4_all_vs_all.tblout.txt').sort_values(['query', 'bitscore'], ascending=[True, False]).set_index('query')

duplicates = []
for query in sorted(set(all_vs_all.index)):
    df = all_vs_all.loc[query]
    top_score = np.round(df.iloc[0]['bitscore'], 1)

    for target, bitscore in df.iloc[1:][['target', 'bitscore']].values:
        if  np.round(bitscore, 1) == top_score:
            duplicates.append((query, target))

assert len(duplicates) == 0

### Mapping to UniProtKB and DB Proka

In [44]:
db_proka_map = pd.read_csv(halh4_folder / 'Halocin_H4_map_to_db_proka.tsv', sep='\t')
db_proka_map = db_proka_map[db_proka_map['fident'] == 1.0].reset_index(drop=True)

target_to_ncbi_accession = {
    query: target.split('@')[1]
    for query, target in db_proka_map[['query', 'target']].values
}
db_proka_map = db_proka_map.set_index('query')

In [45]:
uniprotkb_map = pd.read_csv(halh4_folder / 'Halocin_H4_map_to_UniProtKB.tsv', sep='\t')
uniprotkb_map = uniprotkb_map[uniprotkb_map['fident'] == 1.0].reset_index(drop=True).set_index('query')
uniprotkb_map

,target,qlen,tlen,fident,alnlen,mismatch,qstart,qend,tstart,tend,evalue,bits
query,,,,,,,,,,,,
DUGT01000203.1_24,A0A832U284,340,339,1.0,339,0,1,339,1,339,2.369000e-264,819
NZ_LR881183.1_1455,A0A7G2D9G6,268,267,1.0,267,0,1,267,1,267,5.086000e-212,662
NZ_AOIJ01000063.1_217,L9YSX7,280,279,1.0,279,0,1,279,1,279,2.701000e-221,690
NZ_CP044129.1_261,A0A5Q0UIW4,355,338,1.0,338,0,17,354,1,338,1.716000e-268,833
NZ_SNUH01000002.1_460,A0A8J8AIN5,329,328,1.0,328,0,1,328,1,328,5.483000e-261,808
NZ_JONQ01000011.1_150,A0A832RVS1,350,349,1.0,349,0,1,349,1,349,1.031000e-277,858
NZ_SMGQ01000011.1_424,A0A4R1MYC7,404,403,1.0,403,0,1,403,1,403,2.704000e-316,974
NZ_RIAS01000036.1_4,A0A5M9X370,404,403,1.0,403,0,1,403,1,403,1.690000e-310,958
NZ_MRTX01000012.1_15,A0A1R1GAQ2,409,408,1.0,408,0,1,408,1,408,2.312000e-314,969


### Find NCBI accession for non DB Proka hits

In [46]:
def find_ncbi_assembly_assembly(contig_accession, email='rs1521@ic.ac.uk'):
    Entrez.email = email

    # Step 1: esearch - Find the UID for the contig
    search_handle = Entrez.esearch(db="nucleotide", term=contig_accession)
    search_record = Entrez.read(search_handle)
    search_handle.close()
    
    if not search_record["IdList"]:
        return None
    
    contig_uid = search_record["IdList"][0]

    # Step 2: elink - Find all linked UIDs in the assembly database
    link_handle = Entrez.elink(dbfrom="nucleotide", db="assembly", id=contig_uid)
    link_record = Entrez.read(link_handle)
    link_handle.close()
    
    assembly_links = link_record[0]["LinkSetDb"]
    if not assembly_links:
        return None

    assembly_uids = [link['Id'] for link in assembly_links[0]['Link']]

    # Step 3: esummary - Fetch summaries for all linked assembly UIDs
    summary_handle = Entrez.esummary(db="assembly", id=",".join(assembly_uids))
    summary_record = Entrez.read(summary_handle)
    summary_handle.close()

    # Step 4: Filter summaries to find the RefSeq (GCF_) record
    all_summaries = summary_record['DocumentSummarySet']['DocumentSummary']

    if len(all_summaries) > 0:
        for summary in all_summaries:
            return summary['AssemblyAccession']
    else:
        return None

In [47]:
targets = hits_df['target'].unique()
for i, target in enumerate(targets):
    if target in target_to_ncbi_accession:
        continue
    
    contig_id = re.match(r'^([^\.]+\.[0-9]+)_.+$', target)[1]

    assembly_accession = find_ncbi_assembly_assembly(contig_id)

    if assembly_accession is not None:
        target_to_ncbi_accession[target] = assembly_accession

    print(f'{i+1:,} of {len(targets):,} | {contig_id} | {assembly_accession}')

    time.sleep(0.5)

print(len(target_to_ncbi_accession))

5 of 31 | JAATFT010000037.1 | None
6 of 31 | NZ_CP009517.1 | GCF_000970305.1
7 of 31 | DUGT01000203.1 | None
8 of 31 | NZ_MRTX01000012.1 | GCF_001955925.1
10 of 31 | AOIR01000063.1 | GCF_000337115.1
14 of 31 | WAPO01000149.1 | None
18 of 31 | NZ_RIAS01000036.1 | GCF_003719335.1
21 of 31 | NZ_JABMCC010000121.1 | GCF_013359905.1
22 of 31 | NZ_LR881183.1 | GCF_904067545.1
26 of 31 | CALUHR010000006.1 | GCA_944320495.1
27 of 31 | WANV01000030.1 | None
28 of 31 | NZ_SNUH01000002.1 | GCF_012027355.1
27


### Final dataset

In [48]:
# GenBank is used for this genome in GTDB somehow
target_to_ncbi_accession['AOIR01000063.1_62'] = target_to_ncbi_accession['AOIR01000063.1_62'].replace('GCF_', 'GCA_')

In [ ]:
halh4_df = hits_df.copy().drop(columns=['query']).rename(columns={'target': 'legacy_id'})
halh4_df['assembly_accession'] = halh4_df['legacy_id'].apply(lambda v: target_to_ncbi_accession.get(v))
halh4_df = halh4_df[halh4_df['assembly_accession'].notnull()].reset_index(drop=True)

def set_protein_id(legacy_id):
    if legacy_id in db_proka_map.index:
        return db_proka_map.loc[legacy_id, 'target'].split('@')[0]
    else:
        return legacy_id

halh4_df['protein_id'] = halh4_df['legacy_id'].apply(set_protein_id)
halh4_df['id'] = halh4_df.apply(lambda row: f"{row['protein_id']}@{row['assembly_accession']}", axis=1)
halh4_df = halh4_df.set_index('id')

def set_uniprot_id(legacy_id):
    if legacy_id in uniprotkb_map.index:
        return uniprotkb_map.loc[legacy_id, 'target']
    else:
        return None

halh4_df['uniprot_id'] = halh4_df['legacy_id'].apply(set_uniprot_id)

final_records = []
for record in SeqIO.parse(halh4_out / 'Halocin_H4_GTDB_hits.fasta', 'fasta'):
    if record.id not in halh4_df['legacy_id'].unique():
        continue

    if record.seq[-1] == '*':
        record.seq = record.seq[:-1]

    record.id = halh4_df[halh4_df['legacy_id'] == record.id].index[0]
    record.name = ''
    record.description = ''
    final_records.append(record)

halh4_df = halh4_df[
    ['assembly_accession', 'protein_id'] + [c for c in halh4_df.columns if c not in ('assembly_accession', 'protein_id')]
].drop(columns=['legacy_id'])

halh4_df

,assembly_accession,protein_id,evalue,bits,tstart,tend,domain,gtdb_phylum,gtdb_class,gtdb_order,gtdb_family,gtdb_genus,gtdb_species,uniprot_id
id,,,,,,,,,,,,,,
WP_014732703.1@GCF_000306765.2,GCF_000306765.2,WP_014732703.1,7.295000e-233,723,1,359,Archaea,Halobacteriota,Halobacteria,Halobacteriales,Haloferacaceae,Haloferax,Haloferax mediterranei,Q48236
WP_241433888.1@GCF_000337535.1,GCF_000337535.1,WP_241433888.1,2.720000e-100,340,48,347,Archaea,Halobacteriota,Halobacteria,Halobacteriales,Natrialbaceae,Natrialba,Natrialba aegyptia,None
WP_008458115.1@GCF_000337175.1,GCF_000337175.1,WP_008458115.1,2.408000e-93,320,1,276,Archaea,Halobacteriota,Halobacteria,Halobacteriales,Natrialbaceae,Natrinema,Natrinema gari,L9YSX7
WP_161605605.1@GCF_000337135.1,GCF_000337135.1,WP_161605605.1,1.911000e-42,172,6,156,Archaea,Halobacteriota,Halobacteria,Halobacteriales,Natrialbaceae,Natrialba,Natrialba chahannaoensis,None
NZ_CP009517.1_2169@GCF_000970305.1,GCF_000970305.1,NZ_CP009517.1_2169,1.735000e-28,130,69,300,Archaea,Halobacteriota,Methanosarcinia,Methanosarcinales,Methanosarcinaceae,Methanosarcina,Methanosarcina barkeri A,A0A0E3SN68
NZ_MRTX01000012.1_15@GCF_001955925.1,GCF_001955925.1,NZ_MRTX01000012.1_15,4.309000e-28,129,47,313,Bacteria,Bacillota,Bacilli,Paenibacillales,Paenibacillaceae,Paenibacillus,Paenibacillus sp001955925,A0A1R1GAQ2
WP_247730522.1@GCF_023093535.1,GCF_023093535.1,WP_247730522.1,7.898000e-28,128,59,279,Archaea,Halobacteriota,Halobacteria,Halobacteriales,Natrialbaceae,Halovivax,Halovivax limisalsi,None
AOIR01000063.1_62@GCA_000337115.1,GCA_000337115.1,AOIR01000063.1_62,1.959000e-27,127,60,280,Archaea,Halobacteriota,Halobacteria,Halobacteriales,Natrialbaceae,Natrinema,Natrinema thermotolerans,None
JAGXOR010000056.1_11@GCA_023659785.1,GCA_023659785.1,JAGXOR010000056.1_11,1.959000e-27,127,67,201,Archaea,Halobacteriota,Syntropharchaeia,ANME-1,ANME-1,JAGXOR01,JAGXOR01 sp023659785,None


### Fix protein IDs

After genomes have been downloaded with https://github.com/srom/assembly/

In [ ]:
protein_id_change = {}
for assembly_accession, protein_id in halh4_df[['assembly_accession', 'protein_id']].values:
    inner_folder = None
    for p in (halh4_folder / 'genomes').iterdir():
        if p.name.startswith(assembly_accession):
            inner_folder = p
            break

    assert inner_folder is not None

    fasta_file_gz = inner_folder / f'{inner_folder.name}_protein.faa.gz'
    with gzip.open(fasta_file_gz, mode='rt') as f:
        records = SeqIO.to_dict(SeqIO.parse(f, 'fasta'))

    if protein_id not in records:
        output_tsv = inner_folder / 'mmseqs_halH4_match.m8'

        assert output_tsv.is_file()

        matches = pd.read_csv(output_tsv, sep='\t')
        top_matches = matches[matches['fident'] == 1.0]
        
        if len(top_matches) >= 1:
            new_protein_id = top_matches.iloc[0]['target']
            protein_id_change[(assembly_accession, protein_id)] = new_protein_id
        else:
            protein_id_change[(assembly_accession, protein_id)] = None

protein_id_change

{('GCF_000970305.1', 'NZ_CP009517.1_2169'): 'WP_155396796.1',
 ('GCF_001955925.1', 'NZ_MRTX01000012.1_15'): 'WP_076157611.1',
 ('GCA_000337115.1', 'AOIR01000063.1_62'): None,
 ('GCF_003719335.1', 'NZ_RIAS01000036.1_4'): 'WP_123067460.1',
 ('GCF_013359905.1', 'NZ_JABMCC010000121.1_673'): 'WP_175383825.1',
 ('GCF_904067545.1', 'NZ_LR881183.1_1455'): 'WP_188202341.1',
 ('GCF_012027355.1', 'NZ_SNUH01000002.1_460'): 'WP_167895259.1'}

In [89]:
halh4_copy = halh4_df.reset_index()
final_records_copy = copy.deepcopy(final_records)

for (accession, protein_id), new_protein_id in protein_id_change.items():
    if new_protein_id is None:
        # delete record
        final_records_copy = [r for r in final_records_copy if accession not in r.id]
        halh4_copy = halh4_copy[
            ~(
                (halh4_copy['assembly_accession'] == accession) &
                (halh4_copy['protein_id'] == protein_id)
            )
        ].copy()
    else:
        index = halh4_copy[
            (halh4_copy['assembly_accession'] == accession) &
            (halh4_copy['protein_id'] == protein_id)
        ].index[0]
        
        old_id = halh4_copy.loc[index, 'id']
        new_id = f'{new_protein_id}@{accession}'

        halh4_copy.loc[index, 'protein_id'] = new_protein_id
        halh4_copy.loc[index, 'id'] = new_id

        for record in final_records_copy:
            if record.id == old_id:
                record.id = new_id

halh4_copy = halh4_copy.set_index('id')

In [90]:
with (halh4_out / 'Halocin_H4_dataset.fasta').open('w') as f_out:
    SeqIO.write(final_records_copy, f_out, 'fasta')

halh4_copy.to_csv(halh4_out / 'Halocin_H4_dataset.csv')

### Metadata

In [95]:
gtdb_metadata_all = pd.concat(
    [
        pd.read_csv(data_folder / 'gtdb_r214.1' / 'ar53_metadata_r214.tsv', sep='\t'),
        pd.read_csv(data_folder / 'gtdb_r214.1' / 'bac120_metadata_r214.tsv', sep='\t'),
    ], 
    ignore_index=True,
)
gtdb_metadata_all['assembly_accession'] = gtdb_metadata_all['accession'].apply(lambda v: v[3:])
gtdb_metadata_all = gtdb_metadata_all.set_index('assembly_accession')

gtdb_metadata = gtdb_metadata_all.loc[halh4_copy['assembly_accession'].unique()].copy()
gtdb_metadata_all = None

gtdb_metadata.head()

/opt/homebrew/Caskroom/miniforge/base/envs/amp/lib/python3.8/site-packages/IPython/core/interactiveshell.py:3444: DtypeWarning: Columns (61,63,65,74,82,83) have mixed types.Specify dtype option on import or set low_memory=False.
  exec(code_obj, self.user_global_ns, self.user_ns)


,accession,ambiguous_bases,checkm_completeness,checkm_contamination,checkm_marker_count,checkm_marker_lineage,checkm_marker_set_count,checkm_strain_heterogeneity,coding_bases,coding_density,...,ssu_silva_blast_align_len,ssu_silva_blast_bitscore,ssu_silva_blast_evalue,ssu_silva_blast_perc_identity,ssu_silva_blast_subject_id,ssu_silva_taxonomy,total_gap_length,trna_aa_count,trna_count,trna_selenocysteine_count
assembly_accession,,,,,,,,,,,,,,,,,,,,,
GCF_000306765.2,RS_GCF_000306765.2,0,99.25,0.00,417,f__Halobacteriaceae (UID96),263,0.0,3348782,85.762696,...,1473,2721,0,100,CP001868.1745875.1747347,Archaea;Halobacterota;Halobacteria;Halobactera...,0,19,57,0
GCF_000337535.1,RS_GCF_000337535.1,112,99.42,0.55,417,f__Halobacteriaceae (UID96),263,0.0,3898401,84.410902,...,480,887,0,100,AB663459.1.1472,Archaea;Halobacterota;Halobacteria;Halobactera...,0,19,49,0
GCF_000337175.1,RS_GCF_000337175.1,144,99.30,0.13,417,f__Halobacteriaceae (UID96),263,0.0,3361953,83.553935,...,606,1109,0,99.67,CP003412.577.2050,Archaea;Halobacterota;Halobacteria;Halobactera...,0,19,51,0
GCF_000337135.1,RS_GCF_000337135.1,211,99.74,0.76,417,f__Halobacteriaceae (UID96),263,50.0,3626794,84.162529,...,1551,2861,0,100,AOIN01000039.15.1565,Archaea;Halobacterota;Halobacteria;Halobactera...,0,19,48,0
GCF_000970305.1,RS_GCF_000970305.1,0,99.18,0.65,228,p__Euryarchaeota (UID49),153,0.0,3272668,71.762016,...,1474,2682,0,99.525,AF028692.1.1478,Archaea;Halobacterota;Methanosarcinia;Methanos...,0,19,57,0


In [96]:
gtdb_metadata['domain'] = [halh4_df[halh4_df['assembly_accession'] == a].iloc[0]['domain'] for a in gtdb_metadata.index]
gtdb_metadata['gtdb_phylum'] = [halh4_df[halh4_df['assembly_accession'] == a].iloc[0]['gtdb_phylum'] for a in gtdb_metadata.index]
gtdb_metadata['gtdb_class'] = [halh4_df[halh4_df['assembly_accession'] == a].iloc[0]['gtdb_class'] for a in gtdb_metadata.index]
gtdb_metadata['gtdb_order'] = [halh4_df[halh4_df['assembly_accession'] == a].iloc[0]['gtdb_order'] for a in gtdb_metadata.index]
gtdb_metadata['gtdb_family'] = [halh4_df[halh4_df['assembly_accession'] == a].iloc[0]['gtdb_family'] for a in gtdb_metadata.index]
gtdb_metadata['gtdb_genus'] = [halh4_df[halh4_df['assembly_accession'] == a].iloc[0]['gtdb_genus'] for a in gtdb_metadata.index]
gtdb_metadata['gtdb_species'] = [halh4_df[halh4_df['assembly_accession'] == a].iloc[0]['gtdb_species'] for a in gtdb_metadata.index]

In [97]:
gtdb_metadata.to_csv(halh4_out / 'Halocin_H4_gtdb_metadata.csv')